# projeto Análise de Dados: DataMarts Lavajato

In [ ]:
pip install faker pandas

In [ ]:
def gerar_dias_validos(data):
    return data.weekday() != 5

    

#### mock_data_generator.py

In [1]:
import pandas as pd
from faker import Faker
import random
import uuid
from datetime import datetime, timedelta

# Inicializa o Faker para Português do Brasil
fake = Faker('pt_BR')

# --- CONFIGURAÇÃO ---
# Quantidade de registros a gerar
QTD_CLIENTES = 500
QTD_CARROS = 600  # Alguns clientes têm mais de um carro
QTD_LAVAGENS_FATO = 10000 # Número de registros na tabela fato
DATA_INICIO_SIMULACAO = datetime(2023, 1, 1)
DATA_FIM_SIMULACAO = datetime(2024, 5, 31) # Simula cerca de 1 ano e meio

print(f"Iniciando geração de dados fictícios para Lava Jato...")
print(f"Período: {DATA_INICIO_SIMULACAO.date()} até {DATA_FIM_SIMULACAO.date()}")

# ---------------------------------------------------------
# 1. DIMENSÃO: SERVIÇOS (Dados estáticos e realistas)
# ---------------------------------------------------------
print("Gerando Dimensão Serviços...")
servicos_data = [
    {"Tipo_Servico": "Lavagem Simples", "Categoria_Servico": "Padrão", "Preco_Base": 50.00, "tempo": [45, 60]},
    {"Tipo_Servico": "Lavagem Completa", "Categoria_Servico": "Padrão", "Preco_Base": 80.00, "tempo": [50, 65]},
    {"Tipo_Servico": "Lavagem Motor", "Categoria_Servico": "Especial", "Preco_Base": 100.00, "tempo": [20, 30], "restricao": {"inicio": 12, "fim": 17}},
    {"Tipo_Servico": "Polimento Comercial", "Categoria_Servico": "Estética", "Preco_Base": 250.00, "tempo": [30, 45], "restricao": {"inicio": 9, "fim":16}},
    {"Tipo_Servico": "Higienização Interna", "Categoria_Servico": "Estética", "Preco_Base": 350.00, "tempo": [20, 40],"restricao": {"inicio": 7, "fim": 19}},
    {"Tipo_Servico": "Vitrificação Pintura", "Categoria_Servico": "Premium", "Preco_Base": 1200.00, "tempo":[60, 120], "restricao":{"inicio":7, "fim": 14}},    
]


df_servicos = pd.DataFrame(servicos_data)
df_servicos['ID_Servico'] = [i + 1 for i in range(len(servicos_data))] # IDs sequenciais


# ---------------------------------------------------------
# 2. DIMENSÃO: CARROS (Mistura de Faker e dados estáticos)
# ---------------------------------------------------------
print("Gerando Dimensão Carros...")
# Dados de apoio para carros comuns no Brasil
marcas_modelos = {
    "Fiat": [("Argo", "Pequeno"), ("Mobi", "Pequeno"), ("Toro", "Grande"), ("Strada", "Médio")],
    "Volkswagen": [("Gol", "Pequeno"), ("Polo", "Pequeno"), ("T-Cross", "Médio"), ("Amarok", "Grande")],
    "Chevrolet": [("Onix", "Pequeno"), ("Tracker", "Médio"), ("S10", "Grande"), ("Spin", "Médio")],
    "Hyundai": [("HB20", "Pequeno"), ("Creta", "Médio")],
    "Toyota": [("Corolla", "Médio"), ("Hilux", "Grande"), ("Yaris", "Pequeno")]
}
marcas = list(marcas_modelos.keys())

carros_list = []
for _ in range(QTD_CARROS):
    marca = random.choice(marcas)
    modelo_info = random.choice(marcas_modelos[marca])

    carros_list.append({
        "ID_Carro": str(uuid.uuid4())[:8], # ID curto aleatório
        "Marca": marca,
        "Modelo": modelo_info[0],
        "Categoria": modelo_info[1],
        "Placa": fake.bothify(text='???-####').upper(), # Gera placa antiga (AAA-1234)
        "Ano": random.randint(2010, 2024)
    })
df_carros = pd.DataFrame(carros_list)


# ---------------------------------------------------------
# 3. DIMENSÃO: CLIENTES (Para enriquecer a análise)
# ---------------------------------------------------------
print("Gerando Dimensão Clientes...")
clientes_list = []
for _ in range(QTD_CLIENTES):
    clientes_list.append({
        "ID_Cliente": str(uuid.uuid4())[:8],
        "Nome_Cliente": fake.name(),
        "Cidade": "Palmas",
        "Estado": "TO",
        "Tipo_Cliente": random.choice(["Pessoa Física", "Pessoa Física", "Pessoa Jurídica"]) # Mais PF que PJ
    })
df_clientes = pd.DataFrame(clientes_list)


# ---------------------------------------------------------
# 4. TABELA FATO: LAVAGENS
# ---------------------------------------------------------
print(f"Gerando Tabela Fato Lavagens ({QTD_LAVAGENS_FATO} registros)...")
fato_list = []
id_lavagem = 1

# Preparar IDs para escolha aleatória eficiente
ids_carros = df_carros['ID_Carro'].tolist()
ids_servicos = df_servicos['ID_Servico'].tolist()
precos_servicos = df_servicos.set_index('ID_Servico')['Preco_Base'].to_dict()
map_servicos = df_servicos.set_index('ID_Servico').to_dict('index')

# Mapeamento de Categoria do Carro para Multiplicador de Preço (carro grande paga mais)
somador_categoria = {
    "Pequeno": 30,
    "Médio": 40,
    "Grande": 50 
}

carro_categoria_map = df_carros.set_index('ID_Carro')['Categoria'].to_dict()

current_date = DATA_INICIO_SIMULACAO
dias_totais = (DATA_FIM_SIMULACAO - DATA_INICIO_SIMULACAO).days



# Tenta distribuir as lavagens ao longo dos dias, com sazonalidade
while current_date <= DATA_FIM_SIMULACAO:
    # Sazonalidade simples: mais lavagens no Fim de semana (Sexta, Sábado)
    # E menos nos meses de chuva (ex: Jan, Fev no Sudeste)
    peso_dia = 1.0 if current_date.weekday() in [4,6] else 1.5
    peso_mes = 0.7 if current_date.month in [1, 2, 12] else 1.0

    if current_date.weekday() != 5:
        atual = datetime.combine(current_date, datetime.min.time()).replace(hour=6)
        limite = datetime.combine(current_date, datetime.min.time()).replace(hour=19)
        limite_tolerancia = limite + timedelta(minutes=30)
        
        while atual < limite_tolerancia:
            if random.random() > (0.5 * peso_dia * peso_mes):
                atual += timedelta(minutes=random.randint(20,70))
                continue
            id_servico = random.choice(ids_servicos)
            servico = map_servicos[id_servico]

            duracao = random.randint(servico["tempo"][0], servico["tempo"][1])

            inicio = atual
            fim = inicio + timedelta(minutes=duracao)
            restricao = servico.get('restricao')
            #  REGRA 1: limite do dia + tolerância
            if fim > limite_tolerancia:
                break

            #  verificando se restricao existe
            if not isinstance(restricao, dict):
                restricao = {"inicio": 7, "fim": 19}

            if not (restricao["inicio"] <= inicio.hour < restricao["fim"]):
                continue

            #  carro
            id_carro = random.choice(ids_carros)

            categoria_carro = carro_categoria_map[id_carro]
            preco_base = precos_servicos[id_servico]
            valor_final = round(preco_base + somador_categoria[categoria_carro], 2)


            fato_list.append({
                "ID_Lavagem": id_lavagem,
                "ID_Carro": id_carro,
                "ID_Servico": id_servico,
                "ID_Cliente": random.choice(df_clientes['ID_Cliente'].tolist()),
                "Data_Hora_Inicio": inicio,
                "Data_Hora_Fim": fim,
                "Data_Lavagem": inicio.date(),
                "Valor_Pago": valor_final,
                "Quantidade": 1
            })

            id_lavagem += 1

            #  intervalo inteligente (horário de pico)
            if 9 <= inicio.hour <= 11 or 14 <= inicio.hour <= 17:
                intervalo = random.randint(20, 40)
            else:
                intervalo = random.randint(40, 70)

            atual += timedelta(minutes=duracao)

    current_date += timedelta(days=1)

        

df_fato = pd.DataFrame(fato_list)

# Corrige a quantidade real gerada caso a sazonalidade tenha removido muitos registros
print(f"Total de registros fatais realmente gerados: {len(df_fato)}")

# ---------------------------------------------------------
# 5. SALVAR ARQUIVOS CSV
# ---------------------------------------------------------
print("Salvando arquivos CSV...")
df_servicos.to_csv('dim_servicos.csv', index=False, sep=';', encoding='utf-8-sig')
df_carros.to_csv('dim_carros.csv', index=False, sep=';', encoding='utf-8-sig')
df_clientes.to_csv('dim_clientes.csv', index=False, sep=';', encoding='utf-8-sig')
df_fato.to_csv('fato_lavagens.csv', index=False, sep=';', encoding='utf-8-sig')







Iniciando geração de dados fictícios para Lava Jato...
Período: 2023-01-01 até 2024-05-31
Gerando Dimensão Serviços...
Gerando Dimensão Carros...
Gerando Dimensão Clientes...
Gerando Tabela Fato Lavagens (10000 registros)...
Total de registros fatais realmente gerados: 3635
Salvando arquivos CSV...


In [ ]:
pip install sqlalchemy psycopg2-binary

#### pipeline_etl.py

In [2]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///lavajato_datamart.db')

print('Extração de dados')

df_servicos = pd.read_csv('dim_servicos.csv', sep=';')
df_carros = pd.read_csv('dim_carros.csv', sep=';')
df_clientes = pd.read_csv('dim_clientes.csv', sep=';')
df_fato = pd.read_csv('fato_lavagens.csv', sep=';')

print("Arquivos carregados com sucesso!")


print(" TRANSFORM - Tratando dados...")

#  Padronizar datetime
df_fato['Data_Hora_Inicio'] = pd.to_datetime(df_fato['Data_Hora_Inicio'])

#  Criar DIM_TEMPO baseada no datetime real
data_unicas = df_fato['Data_Hora_Inicio'].drop_duplicates()

df_tempo = pd.DataFrame({'DataHora': data_unicas})

df_tempo['ID_Tempo'] = df_tempo['DataHora'].dt.strftime('%d%m%Y%H%M').astype(int)
df_tempo['Dia'] = df_tempo['DataHora'].dt.day
df_tempo['Mes'] = df_tempo['DataHora'].dt.month
df_tempo['Ano'] = df_tempo['DataHora'].dt.year
df_tempo['Hora'] = df_tempo['DataHora'].dt.hour
df_tempo['Data'] = df_tempo['DataHora'].dt.date
df_tempo['Minuto'] = df_tempo['DataHora'].dt.minute
df_tempo['Dia_da_Semana'] = df_tempo['DataHora'].dt.weekday
df_tempo['Fim_de_Semana'] = (df_tempo['Dia_da_Semana'] >= 5).astype(int)

#  Criar chave estrangeira na fato
df_fato['ID_Tempo'] = df_fato['Data_Hora_Inicio'].dt.strftime('%d%m%Y%H%M').astype(int)

#  (Opcional) remover coluna original
df_fato = df_fato.drop(columns=['Data_Hora_Inicio', 'Data_Lavagem', 'Data_Hora_Fim'])

print("Transformações concluídas!")

print(" LOAD - Enviando para o banco...")

df_servicos.to_sql('dim_servico', engine, if_exists='replace', index=False)
df_carros.to_sql('dim_carro', engine, if_exists='replace', index=False)
df_clientes.to_sql('dim_cliente', engine, if_exists='replace', index=False)
df_tempo.to_sql('dim_tempo', engine, if_exists='replace', index=False)
df_fato.to_sql('fato_lavagens', engine, if_exists='replace', index=False)

print(" ETL concluído com sucesso!")


Extração de dados
Arquivos carregados com sucesso!
 TRANSFORM - Tratando dados...
Transformações concluídas!
 LOAD - Enviando para o banco...
 ETL concluído com sucesso!


#### Data Mart Operacional

In [3]:
print("Criando Data Mart Operacional...")

dm_operacional = df_fato.merge(df_tempo, on='ID_Tempo')

dm_operacional = dm_operacional.groupby(
    ['Data', 'Hora', 'Dia_da_Semana']
).agg({
    'Quantidade': 'sum'
}).reset_index()

dm_operacional.rename(columns={
    'Quantidade': 'Total_Lavagens'
}, inplace=True)

Criando Data Mart Operacional...


#### Data Mart Financeiro

In [4]:
print("Criando Data Mart Financeiro...")

dm_financeiro = df_fato.merge(df_tempo, on='ID_Tempo') \
                      .merge(df_servicos, on='ID_Servico')

dm_financeiro = dm_financeiro.groupby(
    ['Ano', 'Mes', 'Tipo_Servico']
).agg({
    'Valor_Pago': 'sum'
}).reset_index()

dm_financeiro.rename(columns={
    'Valor_Pago': 'Faturamento'
}, inplace=True)

Criando Data Mart Financeiro...


#### Data Mart Clientes

In [5]:
print("Criando Data Mart Clientes...")

# relacionamento indireto via carro (se quiser evoluir depois)
dm_clientes = df_fato.merge(df_carros, on='ID_Carro') \
                    .merge(df_tempo, on='ID_Tempo')

dm_clientes = dm_clientes.groupby(
    ['Categoria', 'Mes']
).agg({
    'ID_Lavagem': 'count'
}).reset_index()

dm_clientes.rename(columns={
    'ID_Lavagem': 'Qtd_Lavagens'
}, inplace=True)

Criando Data Mart Clientes...


#### Data Mart Serviços


In [6]:
print("Criando Data Mart Serviços...")

dm_servicos = df_fato.merge(df_servicos, on='ID_Servico') \
                    .merge(df_tempo, on='ID_Tempo')

dm_servicos = dm_servicos.groupby(
    ['Tipo_Servico', 'Categoria_Servico']
).agg({
    'ID_Lavagem': 'count',
    'Valor_Pago': 'sum'
}).reset_index()

dm_servicos.rename(columns={
    'ID_Lavagem': 'Qtd_Servicos',
    'Valor_Pago': 'Faturamento'
}, inplace=True)

Criando Data Mart Serviços...


#### salvar DataMarts no Banco

In [7]:
print("Salvando Data Marts...")

dm_operacional.to_sql('dm_operacional', engine, if_exists='replace', index=False)
dm_financeiro.to_sql('dm_financeiro', engine, if_exists='replace', index=False)
dm_clientes.to_sql('dm_clientes', engine, if_exists='replace', index=False)
dm_servicos.to_sql('dm_servicos', engine, if_exists='replace', index=False)

Salvando Data Marts...


6